## NLP Home Assignment - 4
### Name: Anushka Sawant
### Matriculation Number: 100006644

In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN, Embedding, RepeatVector
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

2026-05-28 10:56:23.659124: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779965783.972786      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779965784.065702      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779965784.802381      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779965784.802440      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779965784.802444      58 computation_placer.cc:177] computation placer alr

In [2]:
# 1. THE DATA
# ==========================================
english_sentences = [
    "The weather is nice today", "I need water", "Where is station",
    "hello", "good morning", "good evening", "I am hungry",
    "I love you", "The cat is sleeping", "Learning AI is fun"
]
 
german_sentences = [
    "Das wetter ist heute schön", "Ich brauche wasser", "Wo ist der Bahnhof",
    "hallo", "guten morgen", "guten Abend", "Ich habe hunger",
    "Ich liebe dich", "Die katze schläft", "KI zu lernen macht spaß"
]
 

In [3]:
# 2. TOKENIZATION & PADDING
# ==========================================
# Tokenize English (Inputs)
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)
eng_vocab_size = len(eng_tokenizer.word_index) + 1
 
X_seq = eng_tokenizer.texts_to_sequences(english_sentences)
max_eng_len = max(len(seq) for seq in X_seq)
X = pad_sequences(X_seq, maxlen=max_eng_len, padding='post')
 
# Tokenize German (Targets)
ger_tokenizer = Tokenizer()
ger_tokenizer.fit_on_texts(german_sentences)
ger_vocab_size = len(ger_tokenizer.word_index) + 1
 
Y_seq = ger_tokenizer.texts_to_sequences(german_sentences)
max_ger_len = max(len(seq) for seq in Y_seq)
Y = pad_sequences(Y_seq, maxlen=max_ger_len, padding='post')
 
# Neural networks expect a 3D target array for sequence output
Y = Y.reshape(Y.shape[0], Y.shape[1], 1)
 

In [4]:
# 3. BUILD THE HYBRID SEQUENTIAL MODEL
# ==========================================
model = Sequential()
 
# --- ENCODER (Reads the English) ---
model.add(Embedding(input_dim=eng_vocab_size, output_dim=16))
 

model.add(SimpleRNN(32, return_sequences=True))

model.add(LSTM(32))
 
# We repeat that final memory vector for every German word we want to output
model.add(RepeatVector(max_ger_len))
 
# --- DECODER (Writes the German) ---
# return_sequences=True forces the output of a prediction for every single word step
model.add(LSTM(32, return_sequences=True))
model.add(Dense(ger_vocab_size, activation='softmax'))

2026-05-28 10:56:51.858342: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
# 4. COMPILE AND TRAIN
# ==========================================
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
 
model.summary()
 
# Increased epochs slightly because the stacked layers take a bit longer to learn
model.fit(X, Y, epochs=250, verbose=0)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Training model...
Training complete!


In [7]:
# 5. TEST THE TRANSLATOR
# ==========================================
test_sentence = 'I need water'
 
# CRITICAL FIX: Wrap the single string in a list [ ]
test_seq = eng_tokenizer.texts_to_sequences([test_sentence]) 
test_padded = pad_sequences(test_seq, maxlen=max_eng_len, padding='post')
 
# Predict the German sequence
prediction = model.predict(test_padded, verbose=0)
 
# Grab the highest probability word ID for each step
predicted_word_ids = np.argmax(prediction[0], axis=-1)
 
# Reverse dictionary to turn numbers back into German words
ger_index_word = {v: k for k, v in ger_tokenizer.word_index.items()}
 
translated_words = []
for word_id in predicted_word_ids:
    if word_id != 0: # Ignore the padding zeros
        translated_words.append(ger_index_word.get(word_id, ""))
 
# FIXED PRINT: Print the actual full sentence string
print(f"\nEnglish Input : {test_sentence}")
print(f"German Output : {' '.join(translated_words)}")


English Input : I need water
German Output : ich brauche wasser
